# 📦 01 — Generación de Datos Sintéticos

**Responsabilidad:** Crear datasets sintéticos de vehículos eléctricos (EV) y conductores,
con niveles de separación, ruido y perfiles de puntualidad configurables.

**Salidas principales:**
- `vehicles_extended.csv` — Dataset base de vehículos eléctricos
- `vehicles_precision_1.csv` … `vehicles_precision_6.csv` — Datasets con separación creciente
- `dataset_parametrizado.csv` — Dataset personalizado por grupos (puntuales/impuntuales)
- `dataset_final.csv` — Dataset con ruido configurable para experimentos avanzados

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import random

## 2. Generador de Dataset Base (formato original)

Genera un dataset en el **formato original** del proyecto con todas las columnas
necesarias para el clustering: datos del vehículo, batería y perfil del conductor.

In [ ]:
def generar_dataset_formato_original(n, guardar_csv=False, nombre_archivo="dataset_generado.csv"):
    """
    Genera un dataset sintético de n vehículos/conductores en el formato
    original del proyecto.

    Columnas generadas:
    - id: identificador único del registro
    - distance: distancia recorrida desde la última carga (km)
    - available_time_start/end: ventana horaria de disponibilidad para cargar
    - current_charge: carga actual de la batería (kWh)
    - battery_capacity: capacidad total de la batería (kWh)
    - charge_speed: velocidad de carga (kW)
    - discharge_rate: tasa de descarga (kW)
    - driver_age: edad del conductor
    - driver_profession: profesión (worker/student/retired)
    - use_frequency: días por semana de uso
    - origin_distance: distancia al punto de origen habitual (km)
    """

    # Profesiones disponibles para los conductores
    profesiones = ["worker", "student", "retired"]
    datos = []

    for i in range(1, n + 1):

        # --- Selección del tipo de conductor con pesos realistas ---
        # Los trabajadores son el grupo más común (50%), seguidos de estudiantes (30%)
        tipo = random.choices(profesiones, weights=[0.5, 0.3, 0.2])[0]

        # --- Edad según profesión ---
        # Cada profesión tiene un rango de edad representativo
        if tipo == "worker":
            edad = random.randint(25, 55)
        elif tipo == "student":
            edad = random.randint(18, 26)
        else:  # retired
            edad = random.randint(60, 85)

        # --- Horas de disponibilidad ---
        # Los trabajadores tienen horarios más predecibles (mañana/tarde)
        # Los estudiantes y jubilados son más variables
        def hora_random(h1, h2):
            return random.randint(h1, h2)

        if tipo == "worker":
            h_ini = hora_random(7, 9)
            h_fin = hora_random(15, 17)
        elif tipo == "student":
            h_ini = hora_random(8, 20)
            h_fin = hora_random(h_ini + 1, min(h_ini + 12, 23))
        else:  # retired
            h_ini = hora_random(16, 20)
            h_fin = hora_random(h_ini + 1, min(h_ini + 6, 23))

        # --- Distancias según el tipo de conductor ---
        # Los jubilados suelen tener origen más lejano, reflejando menor frecuencia de uso
        if tipo == "worker":
            origin_dist = max(1, int(np.random.normal(8, 3)))
            dist = max(10, int(np.random.normal(120, 40)))
        elif tipo == "student":
            origin_dist = max(1, int(np.random.normal(10, 4)))
            dist = max(10, int(np.random.normal(140, 50)))
        else:
            origin_dist = max(1, int(np.random.normal(20, 6)))
            dist = max(10, int(np.random.normal(180, 60)))

        # --- Frecuencia de uso (días/semana) ---
        if tipo == "worker":
            frecuencia = random.randint(4, 7)
        elif tipo == "student":
            frecuencia = random.randint(2, 5)
        else:
            frecuencia = random.randint(1, 3)

        # --- Parámetros eléctricos del vehículo ---
        battery_capacity = random.choice([80, 100, 120, 150])  # kWh
        current_charge = random.randint(0, battery_capacity)
        charge_speed = random.randint(5, 15)     # kW
        discharge_rate = random.randint(3, 10)   # kW

        datos.append({
            "id": i,
            "distance": dist,
            "available_time_start": h_ini,
            "available_time_end": h_fin,
            "current_charge": current_charge,
            "battery_capacity": battery_capacity,
            "charge_speed": charge_speed,
            "discharge_rate": discharge_rate,
            "driver_age": edad,
            "driver_profession": tipo,
            "use_frequency": frecuencia,
            "origin_distance": origin_dist
        })

    df = pd.DataFrame(datos)

    if guardar_csv:
        df.to_csv(nombre_archivo, index=False)
        print(f"✔ Archivo generado: {nombre_archivo}")

    return df


# --- Ejemplo de uso: generar y guardar el dataset base ---
df_base = generar_dataset_formato_original(200, guardar_csv=True, nombre_archivo="vehicles_extended.csv")
print(df_base.head())

## 3. Generador de Datasets con Separación Creciente

Crea **6 versiones** del dataset base con niveles de separación entre grupos
crecientes. Se usa para evaluar la robustez del clustering.

- **Nivel 1**: mucho ruido y grupos muy mezclados (difícil de separar)
- **Nivel 6**: grupos muy bien diferenciados (fácil de separar)

In [ ]:
def generar_datasets_precision(df_base):
    """
    A partir del dataset base, genera 6 versiones con separación creciente
    entre usuarios puntuales e impuntuales.

    Estrategia:
    1. Calcular el stress_index (ratio tiempo_requerido / ventana_disponible)
    2. Etiquetar como 'unpunctual' a los que superen la mediana
    3. Para cada nivel i: añadir ruido decreciente y amplificar diferencias
       de los impuntuales con un factor de separación creciente

    Parámetros clave:
    - noise_levels: nivel de ruido por versión (mayor → más difícil)
    - separation_factors: multiplicador del stress_index para impuntuales
    """

    df = df_base.copy()

    # --- Feature engineering base ---
    # Calculamos métricas derivadas necesarias para el stress_index
    df["battery_level"] = df["current_charge"] / df["battery_capacity"]
    df["window_length"] = df["available_time_end"] - df["available_time_start"]
    df["required_energy"] = df["battery_capacity"] - df["current_charge"]
    df["required_time"] = df["required_energy"] / df["charge_speed"]
    # Si la ventana es 0, usamos 0.5h para evitar división por cero
    df["stress_index"] = df["required_time"] / df["window_length"].replace(0, 0.5)

    # --- Etiqueta base: unpunctual si stress_index > mediana ---
    median_stress = df["stress_index"].median()
    df["user_type"] = np.where(df["stress_index"] > median_stress, "unpunctual", "punctual")

    np.random.seed(42)

    # Nivel 1: mucho ruido, poca separación → los grupos se mezclan
    # Nivel 6: poco ruido, mucha separación → los grupos son bien distintos
    noise_levels = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
    separation_factors = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0]

    for i in range(6):
        df_mod = df.copy()

        # Añadir ruido gaussiano proporcional a la desviación de cada columna
        for col in ["current_charge", "charge_speed", "available_time_start", "available_time_end", "distance"]:
            scale = df_mod[col].std() * noise_levels[i]
            df_mod[col] += np.random.normal(0, scale, size=len(df_mod))

        # Recalcular features derivados después de aplicar el ruido
        df_mod["battery_level"] = df_mod["current_charge"] / df_mod["battery_capacity"]
        df_mod["window_length"] = df_mod["available_time_end"] - df_mod["available_time_start"]
        df_mod["required_energy"] = df_mod["battery_capacity"] - df_mod["current_charge"]
        df_mod["required_time"] = df_mod["required_energy"] / df_mod["charge_speed"]
        df_mod["stress_index"] = df_mod["required_time"] / df_mod["window_length"].replace(0, 0.5)

        # Amplificar características de los usuarios impuntuales para crear
        # más separación entre grupos (más estrés, menos ventana de tiempo)
        factor = separation_factors[i]
        df_mod.loc[df_mod["user_type"] == "unpunctual", "stress_index"] *= factor
        df_mod.loc[df_mod["user_type"] == "unpunctual", "required_time"] *= factor
        df_mod.loc[df_mod["user_type"] == "unpunctual", "window_length"] /= factor
        df_mod.loc[df_mod["user_type"] == "unpunctual", "battery_level"] *= 0.9  # menor carga

        filename = f"vehicles_precision_{i+1}.csv"
        df_mod.to_csv(filename, index=False)
        print(f"✅ Guardado {filename} (nivel de precisión {i+1})")

    print("\nDatasets generados del nivel 1 (muy mezclados) al nivel 6 (muy separados).")


# Generamos los datasets a partir del dataset base ya creado
generar_datasets_precision(df_base)

## 4. Generador de Dataset Parametrizable (Puntuales vs Impuntuales)

Crea datasets donde se controla **exactamente** cuántos usuarios son puntuales
y cuántos son impuntuales, con perfiles y horarios diferenciados.

In [ ]:
def generar_dataset_parametrizable(
        n_total,
        n_puntuales,
        n_impuntuales,
        guardar_csv=False,
        nombre_archivo="dataset_parametrizado.csv"):
    """
    Genera un dataset controlado donde se especifica exactamente la proporción
    de conductores puntuales e impuntuales.

    Diferencias de perfil:
    - Puntuales: más workers, horarios fijos tempranos (6-9h inicio, 14-17h fin),
                 alta frecuencia de uso (4-7 días/semana), origen cercano
    - Impuntuales: más students/retired, horarios variables (10-18h inicio, 19-23h fin),
                   baja frecuencia (1-4 días/semana), origen más lejano

    Args:
        n_total: número total de registros (debe ser n_puntuales + n_impuntuales)
        n_puntuales: número de conductores puntuales a generar
        n_impuntuales: número de conductores impuntuales a generar
        guardar_csv: si True, guarda el resultado como CSV
        nombre_archivo: nombre del fichero CSV de salida
    """

    if n_puntuales + n_impuntuales != n_total:
        raise ValueError("n_puntuales + n_impuntuales debe ser igual a n_total")

    datos = []

    # --- Funciones auxiliares de generación ---

    def seleccionar_profesion(puntual=True):
        """Los puntuales tienden a ser trabajadores; los impuntuales, estudiantes/jubilados."""
        if puntual:
            return random.choices(["worker", "student", "retired"], weights=[0.7, 0.2, 0.1])[0]
        else:
            return random.choices(["worker", "student", "retired"], weights=[0.2, 0.4, 0.4])[0]

    def edad_segun_profesion(tipo):
        if tipo == "worker":  return random.randint(25, 55)
        elif tipo == "student": return random.randint(18, 26)
        else:                   return random.randint(60, 85)

    def distancias_segun_profesion(tipo):
        if tipo == "worker":
            return max(1, int(np.random.normal(8, 3))), max(10, int(np.random.normal(120, 40)))
        elif tipo == "student":
            return max(1, int(np.random.normal(10, 4))), max(10, int(np.random.normal(140, 50)))
        else:
            return max(1, int(np.random.normal(20, 6))), max(10, int(np.random.normal(180, 60)))

    def parametros_bateria():
        battery_capacity = random.choice([80, 100, 120, 150])
        current_charge = random.randint(0, battery_capacity)
        charge_speed = random.randint(5, 15)
        discharge_rate = random.randint(3, 10)
        return battery_capacity, current_charge, charge_speed, discharge_rate

    # --- Generar usuarios PUNTUALES ---
    # Horarios predecibles: llegan pronto, salen por la tarde, alta frecuencia
    for i in range(1, n_puntuales + 1):
        tipo = seleccionar_profesion(puntual=True)
        edad = edad_segun_profesion(tipo)
        origin_dist, dist = distancias_segun_profesion(tipo)
        battery_capacity, current_charge, charge_speed, discharge_rate = parametros_bateria()
        h_ini = random.randint(6, 9)
        h_fin = random.randint(14, 17)
        frecuencia = random.randint(4, 7)

        datos.append({
            "id": i, "distance": dist,
            "available_time_start": h_ini, "available_time_end": h_fin,
            "current_charge": current_charge, "battery_capacity": battery_capacity,
            "charge_speed": charge_speed, "discharge_rate": discharge_rate,
            "driver_age": edad, "driver_profession": tipo,
            "use_frequency": frecuencia, "origin_distance": origin_dist,
            "puntuality_group": "punctual"
        })

    # --- Generar usuarios IMPUNTUALES ---
    # Horarios tardíos e irregulares, baja frecuencia de uso
    for i in range(n_puntuales + 1, n_total + 1):
        tipo = seleccionar_profesion(puntual=False)
        edad = edad_segun_profesion(tipo)
        origin_dist, dist = distancias_segun_profesion(tipo)
        battery_capacity, current_charge, charge_speed, discharge_rate = parametros_bateria()
        h_ini = random.randint(10, 18)
        h_fin = random.randint(19, 23)
        frecuencia = random.randint(1, 4)

        datos.append({
            "id": i, "distance": dist,
            "available_time_start": h_ini, "available_time_end": h_fin,
            "current_charge": current_charge, "battery_capacity": battery_capacity,
            "charge_speed": charge_speed, "discharge_rate": discharge_rate,
            "driver_age": edad, "driver_profession": tipo,
            "use_frequency": frecuencia, "origin_distance": origin_dist,
            "puntuality_group": "late"
        })

    df = pd.DataFrame(datos)

    if guardar_csv:
        df.to_csv(nombre_archivo, index=False)
        print(f"✔ Archivo generado: {nombre_archivo}")

    return df


# --- Ejemplo: 60 usuarios totales, mitad puntuales y mitad impuntuales ---
df_param = generar_dataset_parametrizable(
    n_total=60, n_puntuales=30, n_impuntuales=30,
    guardar_csv=True, nombre_archivo="dataset_parametrizado.csv"
)
print(df_param.head())

## 5. Generador de Dataset con Ruido Configurable

Versión avanzada que permite inyectar **ruido suave o duro** para simular
escenarios realistas donde los perfiles de usuario no son perfectamente separables.

In [ ]:
def generate_final_dataset(total_samples=400, noise_ratio=0.15, noise_mode="hard"):
    """
    Genera un dataset con ruido configurable para experimentar con la robustez
    de los algoritmos de clustering.

    Perfiles base:
    - Puntuales: origin_distance ≈ 5km, inicio ≈ 7h, fin ≈ 16h,
                 antigüedad del coche baja, categoría premium
    - Impuntuales: origin_distance ≈ 30km, inicio ≈ 11h, fin ≈ 21h,
                   coche más viejo y de baja gama

    Modos de ruido:
    - 'soft': desplaza ligeramente los valores del perfil base
    - 'hard': asigna valores completamente opuestos o extremos
              (simula errores en los datos o usuarios atípicos)

    Args:
        total_samples: número total de conductores a generar
        noise_ratio: fracción de registros con ruido (0.0 = sin ruido, 1.0 = todo ruido)
        noise_mode: 'soft' o 'hard'
    """

    data = []

    for i in range(total_samples):
        # La primera mitad son puntuales, la segunda impuntuales
        is_punctual = i < (total_samples // 2)
        # Determinamos si este registro tendrá ruido inyectado
        is_noisy = random.random() < noise_ratio

        # --- Perfil base según tipo de usuario ---
        if is_punctual:
            prof, o_dist, t_start, t_end, c_age, c_cat, c_cond = (
                "worker", 5, 7, 16, 2, "premium", "excellent"
            )
        else:
            prof, o_dist, t_start, t_end, c_age, c_cat, c_cond = (
                random.choice(["student", "retired"]), 30, 11, 21, 12, "low_end", "poor"
            )

        # --- Inyección de ruido ---
        if is_noisy and noise_mode == "hard":
            # Ruido duro: valores completamente fuera del perfil habitual
            o_dist = random.uniform(80, 150)  # distancias extremas
            t_start = random.uniform(0, 24)   # hora aleatoria
            prof = "student" if is_punctual else "worker"  # profesión invertida
            c_cat = "low_end" if is_punctual else "premium"  # categoría invertida

        # Añadir variabilidad gaussiana al perfil (con o sin ruido)
        data.append({
            "origin_distance": max(0, random.gauss(o_dist, 1.5)),
            "driver_profession": prof,
            "available_time_start": random.gauss(t_start, 0.5),
            "available_time_end": random.gauss(t_end, 0.5),
            "car_age": max(0, random.gauss(c_age, 1.2)),
            "car_category": c_cat,
            "car_condition": c_cond,
            "punctuality_group": "punctual" if is_punctual else "late"
        })

    df = pd.DataFrame(data)
    return df


# --- Ejemplo: 400 muestras con 15% de ruido duro ---
df_noisy = generate_final_dataset(total_samples=400, noise_ratio=0.15, noise_mode="hard")
df_noisy.to_csv("dataset_final.csv", index=False)
print(f"✔ Dataset con ruido generado: {len(df_noisy)} registros")
print(df_noisy["punctuality_group"].value_counts())